In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# 04 \u2014 EM Algorithm from Scratch"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Objectives\n",
        "- Implement Gaussian PDF, E-step, M-step, likelihood, and stopping.\n",
        "- Visualize initial, intermediate, and final clustering.\n",
        "- Avoid `sklearn.mixture.GaussianMixture`."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Mathematical Background\n",
        "EM alternates $q_i(k)\\leftarrow p(z_i=k\\mid x_i,\\theta)$ and responsibility-weighted parameter updates for $\\pi_k$, $\\mu_k$, and $\\Sigma_k$."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Implementation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import sys\n",
        "from pathlib import Path\n",
        "import numpy as np\n",
        "import matplotlib.pyplot as plt\n",
        "\n",
        "PROJECT = Path.cwd()\n",
        "if PROJECT.name == 'notebooks':\n",
        "    PROJECT = PROJECT.parent\n",
        "FIGURES = PROJECT / 'figures'\n",
        "FIGURES.mkdir(exist_ok=True)\n",
        "if str(PROJECT) not in sys.path:\n",
        "    sys.path.insert(0, str(PROJECT))\n",
        "\n",
        "plt.style.use('seaborn-v0_8-whitegrid')\n",
        "COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']\n",
        "RNG = np.random.default_rng(7)\n",
        "\n",
        "def savefig(name):\n",
        "    plt.tight_layout()\n",
        "    plt.savefig(FIGURES / name, dpi=180, bbox_inches='tight', transparent=True)\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Experiments"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from src.gaussian import sample_mixture\n",
        "from src.em import e_step, m_step\n",
        "from src.utils import initialize_parameters\n",
        "true_w=np.array([.4,.35,.25]); true_m=np.array([[-3,0],[0,2.5],[2.8,-.7]])\n",
        "true_c=np.array([[[.7,.2],[.2,.5]],[[.9,-.35],[-.35,.7]],[[.5,.1],[.1,.6]]])\n",
        "X,_=sample_mixture(true_w,true_m,true_c,1000,random_state=RNG)\n",
        "w,m,c=initialize_parameters(X,3,RNG); history=[]; snapshots={}\n",
        "for it in range(1,31):\n",
        "    r, lp = e_step(X,w,m,c); history.append(lp.sum())\n",
        "    if it in [1,5,10,30]: snapshots[it]=r.copy()\n",
        "    w,m,c = m_step(X,r)\n",
        "for it,name in [(1,'initial_clusters.png'),(5,'iteration_5.png'),(10,'iteration_10.png'),(30,'final_clusters.png')]:\n",
        "    plt.figure(figsize=(6,5)); plt.scatter(X[:,0],X[:,1],c=snapshots[it].argmax(1),cmap='viridis',s=12,alpha=.65)\n",
        "    plt.scatter(m[:,0],m[:,1],marker='x',s=120,c='black'); plt.title(f'EM assignments at iteration {it}')\n",
        "    savefig(name)\n",
        "plt.figure(figsize=(6,4)); plt.plot(history,color=COLORS[0],lw=2.5); plt.title('Does observed log-likelihood increase?'); plt.xlabel('iteration'); plt.ylabel('log likelihood')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Observations\n",
        "- The first assignments are crude, but responsibility-weighted updates quickly move centers and covariances toward coherent components.\n",
        "- Log-likelihood should not decrease for correctly implemented EM."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Key Takeaways\n",
        "- EM is coordinate ascent on a likelihood-related objective.\n",
        "- Visual snapshots make the abstract E/M alternation concrete."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Transition to the next notebook\n",
        "Next we stress-test convergence and initialization rather than assuming EM always finds the best solution."
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "pygments_lexer": "ipython3"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}